## Imports

In [27]:
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime
import time
import os        # To check if files exist
from tqdm.notebook import tqdm # To show a progress bar in the notebook

## Constants and Game IDs

In [28]:
# --- Constants ---
BASE_URL = "https://api-web.nhle.com/v1/gamecenter/"
SEASON = "2023"
GAME_TYPE = "02"  # 01 = Preseason, 02 = Regular Season, 03 = Playoffs
NUM_GAMES = 1312
DATA_FILE_NAME = "nhl_raw_plays.parquet" # Parquet is faster and smaller than CSV

# --- Generate all game IDs ---
# This creates a list of strings like ["2023020001", "2023020002", ..., "2023021312"]
game_ids = [f"{SEASON}{GAME_TYPE}{str(i).zfill(4)}" for i in range(1, NUM_GAMES + 1)]

print(f"Generated {len(game_ids)} game IDs.")
print(f"First ID: {game_ids[0]}, Last ID: {game_ids[-1]}")

Generated 1312 game IDs.
First ID: 2023020001, Last ID: 2023021312


## Data Retrieval

In [29]:
def getGameData(game_id):

# Fetches play-by-play data for a given game ID

    # Create URL
    url = f"{BASE_URL}{game_id}/play-by-play"

    try:
        # Call the API
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes

        # Extract JSON data
        data = response.json()
        plays = data.get("plays", [])
        return plays

    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for game {game_id}: {e}")
        return None # return None if there's an error

In [30]:
all_plays_data = []

# Check if data file exists
if not os.path.exists(DATA_FILE_NAME):
    print(f"Data file {DATA_FILE_NAME} does not exist. Starting data collection...")

    # Loop through all game IDs with progress bar
    for game_id in tqdm(game_ids, desc="Fetching data"):
        plays = getGameData(game_id)

        # Add plays to all_plays_data
        if plays:
            all_plays_data.extend(plays)

    print(f"\nTotal plays collected: {len(all_plays_data)}")

    # Convert to DataFrame
    df = pd.DataFrame(all_plays_data)
    
    # Save to Parquet file
    df.to_parquet(DATA_FILE_NAME, index=False)
    print(f"Data saved to {DATA_FILE_NAME}")
else:
    print(f"Loading data from {DATA_FILE_NAME}...")
    df = pd.read_parquet(DATA_FILE_NAME)
    print(f"Loaded {len(df)} plays from {DATA_FILE_NAME}")
    
# Display first few rows of the DataFrame
print(f"\nDataFrame shape: {df.shape}")
df.head()

Loading data from nhl_raw_plays.parquet...
Loaded 413676 plays from nhl_raw_plays.parquet

DataFrame shape: (413676, 11)


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
0,102,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,left,520,period-start,8,None,None
1,101,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,left,502,faceoff,9,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
2,8,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:35,19:25,1551,left,516,stoppage,15,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
3,103,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:35,19:25,1551,left,502,faceoff,17,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
4,9,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:48,19:12,1551,left,503,hit,20,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None


## Data Cleaning

1. Filter DataFrame for shots
2. Flatten Nested columns (period and details)

In [31]:
# identify all unique events in the data
df["typeDescKey"].unique()

array(['period-start', 'faceoff', 'stoppage', 'hit', 'giveaway',
       'shot-on-goal', 'takeaway', 'missed-shot', 'blocked-shot', 'goal',
       'penalty', 'delayed-penalty', 'period-end', 'game-end',
       'shootout-complete', 'failed-shot-attempt'], dtype=object)

In [32]:
# print original shape of df
print(f"Original shape of df: {df.shape}")
# create list of events that we care about from json data
shot_events = ['shot-on-goal', 'missed-shot', 'blocked-shot', 'goal']

# create new dataframe with only the shot events
df_shots = df[df["typeDescKey"].isin(shot_events)].copy()

# print new shafe of df_shots
print(f"New shape of df_shots: {df_shots.shape}")
df_shots.head()

Original shape of df: (413676, 11)
New shape of df_shots: (160123, 11)


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
6,63,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:01,18:59,1551,left,506,shot-on-goal,22,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
7,151,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:10,18:50,1551,left,506,shot-on-goal,23,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
9,70,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:47,18:13,1551,left,506,shot-on-goal,31,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
16,152,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",02:57,17:03,1551,left,506,shot-on-goal,48,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
18,95,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",03:51,16:09,1551,left,507,missed-shot,60,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None


In [33]:
# flatten nested columns

# flatten the details column
df_details_flat = pd.json_normalize(df_shots["details"])

# flatten the period column
df_period_flat = pd.json_normalize(df_shots["periodDescriptor"])

# reset indices of dataframes to ensure they can join correctly
df_shots.reset_index(drop=True, inplace=True)
df_details_flat.reset_index(drop=True, inplace=True)
df_period_flat.reset_index(drop=True, inplace=True)

# join the flattened columns with the original dataframe
df_shots_clean = pd.concat([df_shots, df_details_flat, df_period_flat], axis=1)

# drop the original nested columns
df_shots_clean.drop(columns=["details", "periodDescriptor"], inplace=True)

# rename the 'number' column (from periodDescriptor) to 'period'
df_shots_clean.rename(columns={"number": "period"}, inplace=True)

print(f"Shape after flattening: {df_shots_clean.shape}")
for col in df_shots_clean.columns:
    print(col)
df_shots_clean.head()

Shape after flattening: (160123, 49)
eventId
timeInPeriod
timeRemaining
situationCode
homeTeamDefendingSide
typeCode
typeDescKey
sortOrder
pptReplayUrl
assist1PlayerId
assist1PlayerTotal
assist2PlayerId
assist2PlayerTotal
awaySOG
awayScore
blockingPlayerId
committedByPlayerId
descKey
discreteClip
discreteClipFr
drawnByPlayerId
duration
eventOwnerTeamId
goalieInNetId
highlightClip
highlightClipFr
highlightClipSharingUrl
highlightClipSharingUrlFr
hitteePlayerId
hittingPlayerId
homeSOG
homeScore
losingPlayerId
playerId
reason
scoringPlayerId
scoringPlayerTotal
secondaryReason
servedByPlayerId
shootingPlayerId
shotType
typeCode
winningPlayerId
xCoord
yCoord
zoneCode
maxRegulationPeriods
period
periodType


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,pptReplayUrl,assist1PlayerId,...,shootingPlayerId,shotType,typeCode,winningPlayerId,xCoord,yCoord,zoneCode,maxRegulationPeriods,period,periodType
0,63,01:01,18:59,1551,left,506,shot-on-goal,22,None,NaN,...,8478178.0,wrist,None,None,58.0,-25.0,O,3,1,REG
1,151,01:10,18:50,1551,left,506,shot-on-goal,23,None,NaN,...,8478010.0,tip-in,None,None,81.0,8.0,O,3,1,REG
2,70,01:47,18:13,1551,left,506,shot-on-goal,31,None,NaN,...,8479661.0,snap,None,None,55.0,30.0,O,3,1,REG
3,152,02:57,17:03,1551,left,506,shot-on-goal,48,None,NaN,...,8479591.0,wrist,None,None,58.0,-30.0,O,3,1,REG
4,95,03:51,16:09,1551,left,507,missed-shot,60,None,NaN,...,8476887.0,wrist,None,None,-63.0,33.0,O,3,1,REG


In [34]:
# remove some unneccesary columns
df_shots_clean.drop(columns=["highlightClipSharingUrl", "highlightClip", "highlightClipSharingUrlFr",
 "highlightClipFr", "discreteClip", "discreteClipFr", "pptReplayUrl"], inplace = True)

df_shots_clean.head()

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,assist1PlayerId,assist1PlayerTotal,...,shootingPlayerId,shotType,typeCode,winningPlayerId,xCoord,yCoord,zoneCode,maxRegulationPeriods,period,periodType
0,63,01:01,18:59,1551,left,506,shot-on-goal,22,NaN,NaN,...,8478178.0,wrist,None,None,58.0,-25.0,O,3,1,REG
1,151,01:10,18:50,1551,left,506,shot-on-goal,23,NaN,NaN,...,8478010.0,tip-in,None,None,81.0,8.0,O,3,1,REG
2,70,01:47,18:13,1551,left,506,shot-on-goal,31,NaN,NaN,...,8479661.0,snap,None,None,55.0,30.0,O,3,1,REG
3,152,02:57,17:03,1551,left,506,shot-on-goal,48,NaN,NaN,...,8479591.0,wrist,None,None,58.0,-30.0,O,3,1,REG
4,95,03:51,16:09,1551,left,507,missed-shot,60,NaN,NaN,...,8476887.0,wrist,None,None,-63.0,33.0,O,3,1,REG


In [35]:
# create is_goal column
df_shots_clean["is_goal"] = np.where(df_shots_clean["typeDescKey"] == "goal", 1, 0)

print("Goal Counts (1=Goal, 0=No Goal):")
print(df_shots_clean["is_goal"].value_counts())

Goal Counts (1=Goal, 0=No Goal):
is_goal
0    151855
1      8268
Name: count, dtype: int64


### Standardize Coordinates

In [36]:
print("Standardizing coordinates...")

# ensure that the x and y coordinates are numeric
df_shots_clean["xCoord"] = pd.to_numeric(df_shots_clean["xCoord"])
df_shots_clean["yCoord"] = pd.to_numeric(df_shots_clean["yCoord"])

# find all shots that were taken at the "negative" net (xCoord < 0)
flip_mask = df_shots_clean["xCoord"] < 0

# for these rows, multiply the xCoord and yCoord by -1
df_shots_clean.loc[flip_mask, "xCoord"] = df_shots_clean.loc[flip_mask, "xCoord"] * -1
df_shots_clean.loc[flip_mask, "yCoord"] = df_shots_clean.loc[flip_mask, "yCoord"] * -1

print("Coordinates Standardized. All shots are now aimed at the positive-x net.")

# check the results
print("\n--- Original Negative-x Shots (Now Flipped) ---")
# find a goal from period 3 (eventId = 740) to see it's flipped
print(df_shots_clean[df_shots_clean["eventId"] == 740][['period', 'xCoord', 'yCoord']])

print("\n--- Original Positive-x Shots (No Change) ---")
# find goal from period 1 (eventId = 154) to see it's unchanged
print(df_shots_clean[df_shots_clean["eventId"] == 154][['period', 'xCoord', 'yCoord']])



Standardizing coordinates...
Coordinates Standardized. All shots are now aimed at the positive-x net.

--- Original Negative-x Shots (Now Flipped) ---
        period  xCoord  yCoord
76           3    81.0    -4.0
470          2    62.0   -26.0
1225         3    73.0   -10.0
3876         2    68.0    20.0
4703         2    60.0   -28.0
...        ...     ...     ...
154117       2    75.0    -2.0
154990       3    54.0    -1.0
156099       2    82.0     1.0
156913       2    76.0    -2.0
157740       3    61.0    20.0

[166 rows x 3 columns]

--- Original Positive-x Shots (No Change) ---
        period  xCoord  yCoord
12           1    50.0   -16.0
157          1    81.0     5.0
1452         2    87.0     5.0
2153         1    54.0    25.0
2387         1    82.0    -2.0
...        ...     ...     ...
158364       1    60.0    34.0
158831       1    73.0    12.0
158954       1    35.0    13.0
159246       1    82.0   -15.0
159366       1    77.0     3.0

[456 rows x 3 columns]


### Missing Data

In [37]:
# check how many rows are missing a 'shotType'
missing_shots = df_shots_clean["shotType"].isnull().sum()
print(f"Number of missing shotTypes: {missing_shots}")

Number of missing shotTypes: 44691


In [38]:
# fill missing ('NaN') values with the string 'unknown'
df_shots_clean["shotType"].fillna("unknown", inplace=True)

print("Filled missing shot types with 'unknown'")

# check value counts again
print("\nValue counts of shotType:")
print(df_shots_clean["shotType"].value_counts())

Filled missing shot types with 'unknown'

Value counts of shotType:
shotType
wrist           63244
unknown         44691
snap            16062
slap            13554
tip-in          10102
backhand         8284
deflected        2327
wrap-around       907
poke              434
bat               429
between-legs       80
cradle              9
Name: count, dtype: int64


In [40]:
for col in df_shots_clean.columns:
    print(col)

df_shots_clean.head()

eventId
timeInPeriod
timeRemaining
situationCode
homeTeamDefendingSide
typeCode
typeDescKey
sortOrder
assist1PlayerId
assist1PlayerTotal
assist2PlayerId
assist2PlayerTotal
awaySOG
awayScore
blockingPlayerId
committedByPlayerId
descKey
drawnByPlayerId
duration
eventOwnerTeamId
goalieInNetId
hitteePlayerId
hittingPlayerId
homeSOG
homeScore
losingPlayerId
playerId
reason
scoringPlayerId
scoringPlayerTotal
secondaryReason
servedByPlayerId
shootingPlayerId
shotType
typeCode
winningPlayerId
xCoord
yCoord
zoneCode
maxRegulationPeriods
period
periodType
is_goal


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,assist1PlayerId,assist1PlayerTotal,...,shotType,typeCode,winningPlayerId,xCoord,yCoord,zoneCode,maxRegulationPeriods,period,periodType,is_goal
0,63,01:01,18:59,1551,left,506,shot-on-goal,22,NaN,NaN,...,wrist,None,None,58.0,-25.0,O,3,1,REG,0
1,151,01:10,18:50,1551,left,506,shot-on-goal,23,NaN,NaN,...,tip-in,None,None,81.0,8.0,O,3,1,REG,0
2,70,01:47,18:13,1551,left,506,shot-on-goal,31,NaN,NaN,...,snap,None,None,55.0,30.0,O,3,1,REG,0
3,152,02:57,17:03,1551,left,506,shot-on-goal,48,NaN,NaN,...,wrist,None,None,58.0,-30.0,O,3,1,REG,0
4,95,03:51,16:09,1551,left,507,missed-shot,60,NaN,NaN,...,wrist,None,None,63.0,-33.0,O,3,1,REG,0


## Feature Engineering